# 📊 Módulo 06 - Notebook 03: Métricas ARPU y Churn Rate

## 📈 KPIs Fundamentales para Negocios de suscripción y retail

**Libro:** Saliendo de lo Pandito  
**Módulo:** 06 - Agregaciones y Métricas KPI  
**Duración estimada:** 60 minutos  
**Dificultad:** 🟠 Intermedio-Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Calcular** ARPU (Average Revenue Per User)  
✅ **Medir** Churn Rate (tasa de cancelación)  
✅ **Entender** LTV (Lifetime Value)  
✅ **Analizar** cohorts de clientes  
✅ **Aplicar** métricas SaaS/retail en Pandas

---

## 📋 Pre-requisitos

* ✅ Notebooks 06_01 y 06_02 completados
* ✅ Conocimiento de agregaciones y groupby
* ✅ Familiaridad con métricas de negocio

---

## 📚 Contenido

1. Teoría de ARPU
2. Cálculo de ARPU
3. Teoría de Churn Rate
4. Cálculo de Churn Rate
5. LTV (Lifetime Value)
6. Análisis de Cohorts
7. Caso Integrador: Dashboard de Métricas

---

## 💡 Por qué importa

**Métricas críticas para SaaS, streaming, telecomunicaciones, retail:**

* 💰 **ARPU:** ¿Cuánto genera cada cliente?
* 📉 **Churn:** ¿Cuántos clientes se van?
* 🔄 **LTV:** ¿Cuánto vale un cliente en su vida útil?

**Sin estas métricas, no sabes si tu negocio es sostenible.**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    df_ventas['mes'] = df_ventas['fecha'].dt.to_period('M')
    
    # Simular métricas de clientes/suscriptores
    # Usamos sucursales como "clientes" para el análisis
    np.random.seed(42)
    
    # Simular visitas/transacciones por sucursal
    df_ventas['transacciones'] = np.maximum(1, (df_ventas['ventas'] / 500).round(0).astype(int))
    df_ventas['ticket_promedio'] = df_ventas['ventas'] / df_ventas['transacciones']
    
    # Simular clientes activos por sucursal y mes
    df_ventas['clientes_activos'] = np.random.randint(50, 200, len(df_ventas))
    df_ventas['arpu'] = df_ventas['ventas'] / df_ventas['clientes_activos']
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {len(df_ventas):,}")
    print(f"   📅 Período: {df_ventas['fecha'].min().strftime('%Y-%m-%d')} a {df_ventas['fecha'].max().strftime('%Y-%m-%d')}")
    print(f"   🏪 Sucursales: {df_ventas['sucursal_id'].nunique()}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n📋 Métricas disponibles:")
    print(f"   • ventas: Ingresos por sucursal/mes")
    print(f"   • clientes_activos: Clientes por sucursal/mes")
    print(f"   • arpu: Average Revenue Per User")
    print(f"   • transacciones: Número de transacciones")
    print(f"   • ticket_promedio: Venta promedio por transacción")
    
    print(f"\n🎯 Este notebook usará datos REALES para calcular KPIs")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Métricas de Clientes: ARPU, Churn y LTV

### 💰 ARPU (Average Revenue Per User)

**Definición:** Ingreso promedio por usuario/cliente en un período.

**Fórmula:**
```
ARPU = Ingresos Totales / Número de Usuarios Activos
```

**Ejemplo:**
```
Ingresos del mes: $100,000
Usuarios activos:  1,000
───────────────────────
ARPU:             $100 por usuario
```

**Uso típico:**
* SaaS (Software as a Service)
* Telecomunicaciones
* Streaming (Netflix, Spotify)
* Retail con programa de fidelización

**👉 Variante:** **ARPA** (Average Revenue Per Account) - usado cuando una cuenta puede tener múltiples usuarios.

---

### 📉 Churn Rate (Tasa de Cancelación)

**Definición:** Porcentaje de clientes que dejan de ser activos en un período.

**Fórmula:**
```
Churn Rate = (Clientes perdidos / Clientes al inicio) × 100
```

**Ejemplo:**
```
Clientes al inicio del mes:  1,000
Clientes que cancelaron:       50
─────────────────────────────────
Churn Rate:                    5%
```

**Interpretación:**
* ✅ **< 5% mensual:** Excelente retención (SaaS B2B)
* 🟡 **5-10% mensual:** Aceptable (SaaS B2C)
* ⚠️ **> 10% mensual:** Problema serio de retención

**👉 Inversa:** **Retention Rate** = 100% - Churn Rate

---

### 🔄 LTV (Lifetime Value)

**Definición:** Valor total que un cliente genera durante toda su relación con la empresa.

**Fórmula simplificada:**
```
LTV = ARPU / Churn Rate
```

**Fórmula completa:**
```
LTV = (ARPU × Margen Bruto %) / Churn Rate
```

**Ejemplo:**
```
ARPU:         $100/mes
Churn Rate:   5%/mes  (0.05)
───────────────────────
LTV:          $100 / 0.05 = $2,000
```

**Interpretación:** Cada cliente vale $2,000 en promedio durante su vida.

---

### 🎯 Regla de Oro: LTV > 3 × CAC

**CAC** (Customer Acquisition Cost) = Costo de adquirir un cliente

**Regla:** El LTV debe ser **al menos 3 veces** el CAC para que el negocio sea sostenible.

**Ejemplo:**
```
LTV:  $2,000
CAC:    $500
────────────
Ratio: 4:1  ✅ Saludable
```

---

### 💼 Cálculo en Pandas

```python
# ARPU por mes
arpu = df.groupby('mes').apply(
    lambda x: x['ingresos'].sum() / x['clientes_activos'].mean()
)

# Churn Rate
clientes_inicio = df.groupby('mes')['cliente_id'].nunique()
clientes_fin = df.groupby('mes')['cliente_id'].nunique().shift(-1)
churn = (clientes_inicio - clientes_fin) / clientes_inicio * 100

# LTV
ltv = arpu / (churn / 100)
```

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("📊 ARPU, CHURN RATE Y LTV")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

print("\n🎯 En este notebook aprenderás:")
print("  • ARPU: Average Revenue Per User")
print("  • Churn Rate: Tasa de cancelación de clientes")
print("  • LTV: Lifetime Value (valor de vida del cliente)")
print("  • Análisis de cohorts y retención")

print("\n📖 Métodos clave:")
print("  - arpu = ingresos / clientes_activos")
print("  - churn = (perdidos / inicio) * 100")
print("  - ltv = arpu / churn_rate")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 📊 ARPU y Churn Rate con datos reales de Los Andes Market

### 🏪 Adaptando métricas SaaS a retail

Los Andes Market es una cadena de retail, no un SaaS. Pero podemos adaptar las métricas:

| Métrica SaaS | Adaptación a Los Andes Market |
|-------------|------------------------------|
| **ARPU** (Revenue/User) | Ventas / Clientes activos por sucursal |
| **Churn Rate** | Sucursales que dejan de reportar ventas |
| **LTV** | Valor promedio de una sucursal en su vida útil |
| **Cohortes** | Sucursales agrupadas por año de apertura |

---

### 📊 Simulación de clientes

En la celda de carga, simulamos:
```python
df_ventas['transacciones'] = max(1, ventas / 500)  # ~500 por transacción
df_ventas['ticket_promedio'] = ventas / transacciones
df_ventas['clientes_activos'] = randint(50, 200)  # 50-200 clientes por sucursal/mes
df_ventas['arpu'] = ventas / clientes_activos
```

---

### 💡 Preguntas de negocio

* ¿Qué zona tiene el mayor ARPU (ingreso por cliente)?
* ¿Cuál sucursal atiende más clientes?
* ¿Cómo evoluciona el ticket promedio mes a mes?
* ¿Hay sucursales con alta facturación pero pocos clientes (VIP)?

In [0]:
import pandas as pd

print("📊 ARPU Y CHURN RATE CON DATOS REALES DE LOS ANDES MARKET")
print("="*70)

if USAR_DATOS_REALES and df_ventas is not None:
    print("\n1️⃣  ARPU POR ZONA: Ingreso promedio por cliente")
    print("-"*70)

    arpu_zona = df_ventas.groupby('zona').agg({
        'ventas': 'sum',
        'clientes_activos': 'sum',
        'transacciones': 'sum'
    })
    arpu_zona['arpu'] = (arpu_zona['ventas'] / arpu_zona['clientes_activos']).round(2)
    arpu_zona['ticket_promedio'] = (arpu_zona['ventas'] / arpu_zona['transacciones']).round(2)
    arpu_zona = arpu_zona.sort_values('arpu', ascending=False)
    print("\n   ARPU por zona (ranking):")
    print(arpu_zona[['ventas', 'clientes_activos', 'arpu', 'ticket_promedio']].round(0))

    print("\n" + "="*70)
    print("\n2️⃣  ARPU POR SUCURSAL Y AÑO")
    print("-"*70)

    arpu_año = df_ventas.groupby(['año', 'sucursal_nombre'])['arpu'].mean().unstack('sucursal_nombre')
    print("\n   ARPU promedio por año y sucursal:")
    print(arpu_año.round(0))

    print("\n" + "="*70)
    print("\n3️⃣  EVOLUCIÓN DEL ARPU MENSUAL")
    print("-"*70)

    arpu_mes = df_ventas.groupby(['año', 'mes'])['arpu'].mean().reset_index()
    arpu_mes['periodo'] = arpu_mes['año'].astype(str) + '-' + arpu_mes['mes'].astype(str).str.zfill(2)
    print("\n   ARPU promedio mensual (todos los años):")
    print(arpu_mes[['periodo', 'arpu']].head(15).round(2))

    print("\n" + "="*70)
    print("\n4️⃣  CHURN SIMULADO: Sucursales que dejaron de reportar")
    print("-"*70)

    # Simular churn: una sucursal "churnea" si un mes no tiene ventas
    activas_por_mes = df_ventas.groupby(['año', 'mes'])['sucursal_id'].nunique()
    print("\n   Sucursales activas por mes:")
    print(activas_por_mes.head(15))

    total_sucursales = df_ventas['sucursal_id'].nunique()
    min_activas = activas_por_mes.min()
    churn_simulado = (1 - min_activas / total_sucursales) * 100
    print(f"\n   Total de sucursales: {total_sucursales}")
    print(f"   Mínimo activas en un mes: {min_activas}")
    print(f"   Churn simulado: {churn_simulado:.1f}%")

    print("\n" + "="*70)
    print("\n5️⃣  LTV SIMULADO: Valor de vida de una sucursal")
    print("-"*70)

    arpu_promedio = df_ventas['arpu'].mean()
    # Asumiendo que una sucursal "vive" en promedio 36 meses
    vida_meses = 36
    ltv_sucursal = arpu_promedio * vida_meses * df_ventas['clientes_activos'].mean()
    print(f"\n   ARPU promedio: ${arpu_promedio:.2f}")
    print(f"   Clientes activos promedio: {df_ventas['clientes_activos'].mean():.0f}")
    print(f"   Vida estimada: {vida_meses} meses")
    print(f"   LTV por sucursal: ${ltv_sucursal:,.0f}")

    print("\n" + "="*70)
    print("\n6️⃣  DASHBOARD DE KPIs POR SUCURSAL")
    print("-"*70)

    kpi_sucursal = df_ventas.groupby('sucursal_nombre').agg(
        ventas_total=('ventas', 'sum'),
        clientes_total=('clientes_activos', 'sum'),
        arpu_prom=('arpu', 'mean'),
        ticket_prom=('ticket_promedio', 'mean'),
        transacciones_total=('transacciones', 'sum')
    ).round(0).sort_values('ventas_total', ascending=False)
    print("\n   📋 Dashboard de KPIs por sucursal:")
    print(kpi_sucursal)
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

## 📚 Teoría: ARPU y Churn Rate

### 💰 ARPU (Average Revenue Per User)

**Definición:** Ingreso promedio por usuario en un período

**Fórmula:**
```
ARPU = Total Revenue / Total Active Users
```

**Ejemplo:**
* Netflix tiene 200M suscriptores
* Revenue trimestral: $8,000M
* ARPU trimestral = $8,000M / 200M = **$40 por usuario**
* ARPU mensual = $40 / 3 = **$13.33 por usuario/mes**

---

### 📉 Churn Rate (Tasa de Cancelación)

**Definición:** % de clientes que cancelan en un período

**Fórmula:**
```
Churn Rate = (Usuarios Lost / Usuarios Inicio Período) × 100
```

**Ejemplo:**
* Enero: 10,000 usuarios activos
* Durante Enero: 500 cancelaciones
* Churn Rate = (500 / 10,000) × 100 = **5%**

---

### 🔄 LTV (Lifetime Value)

**Definición:** Valor total que genera un cliente durante su vida útil

**Fórmula Simplificada:**
```
LTV = ARPU / Churn Rate
```

**Ejemplo:**
* ARPU mensual: $50
* Churn Rate mensual: 5% (0.05)
* LTV = $50 / 0.05 = **$1,000**

**Interpretación:** Un cliente promedio genera $1,000 en su vida útil

---

### 🎯 Benchmarks

| Métrica | B2C SaaS | B2B SaaS | Streaming |
|----------|----------|----------|-----------|
| ARPU mensual | $10-30 | $50-500 | $10-20 |
| Churn Rate mensual | 5-7% | 2-3% | 3-5% |
| LTV | $150-$500 | $2K-$20K | $200-$400 |

In [0]:
import pandas as pd
import numpy as np

print("💰 CÁLCULO DE ARPU Y CHURN RATE")
print("="*70)

# Datos de suscriptores por mes
mes_data = pd.DataFrame({
    'Mes': ['Ene', 'Feb', 'Mar', 'Abr', 'May'],
    'Usuarios_Inicio': [10000, 10300, 10800, 11200, 11500],
    'Nuevos_Usuarios': [800, 1000, 900, 850, 900],
    'Cancelaciones': [500, 500, 500, 550, 600],
    'Revenue': [500000, 515000, 540000, 560000, 575000]
})

print("\n📊 Datos mensuales:")
print(mes_data)

print("\n" + "="*70)
print("\n1️⃣  CALCULAR USUARIOS AL FINAL DEL MES")
print("-"*70)

mes_data['Usuarios_Final'] = (mes_data['Usuarios_Inicio'] + 
                               mes_data['Nuevos_Usuarios'] - 
                               mes_data['Cancelaciones'])

print(mes_data[['Mes', 'Usuarios_Inicio', 'Nuevos_Usuarios', 'Cancelaciones', 'Usuarios_Final']])

print("\n" + "-"*70)
print("\n2️⃣  CALCULAR ARPU MENSUAL")
print("-"*70)

mes_data['ARPU'] = (mes_data['Revenue'] / mes_data['Usuarios_Inicio']).round(2)

print(mes_data[['Mes', 'Revenue', 'Usuarios_Inicio', 'ARPU']])
print(f"\n💰 ARPU promedio: ${mes_data['ARPU'].mean():.2f} por usuario/mes")

print("\n" + "-"*70)
print("\n3️⃣  CALCULAR CHURN RATE")
print("-"*70)

mes_data['Churn_Rate_%'] = (mes_data['Cancelaciones'] / mes_data['Usuarios_Inicio'] * 100).round(2)

print(mes_data[['Mes', 'Usuarios_Inicio', 'Cancelaciones', 'Churn_Rate_%']])
print(f"\n📉 Churn promedio: {mes_data['Churn_Rate_%'].mean():.2f}%")

print("\n" + "-"*70)
print("\n4️⃣  CALCULAR CRECIMIENTO NETO")
print("-"*70)

mes_data['Crecimiento_Neto'] = mes_data['Nuevos_Usuarios'] - mes_data['Cancelaciones']
mes_data['Tasa_Crecimiento_%'] = (mes_data['Crecimiento_Neto'] / mes_data['Usuarios_Inicio'] * 100).round(2)

print(mes_data[['Mes', 'Nuevos_Usuarios', 'Cancelaciones', 'Crecimiento_Neto', 'Tasa_Crecimiento_%']])

print("\n" + "-"*70)
print("\n5️⃣  CALCULAR LTV (LIFETIME VALUE)")
print("-"*70)

arpu_promedio = mes_data['ARPU'].mean()
churn_promedio = mes_data['Churn_Rate_%'].mean() / 100  # Convertir a decimal

ltv = arpu_promedio / churn_promedio

print(f"ARPU promedio mensual:   ${arpu_promedio:.2f}")
print(f"Churn Rate promedio:     {churn_promedio*100:.2f}%")
print(f"\n🎯 LTV (Lifetime Value):   ${ltv:.2f}")
print(f"\nInterpretación: Un cliente promedio genera ${ltv:.2f} durante su vida útil")

print("\n" + "="*70)
print("✅ ARPU, Churn y LTV calculados")

In [0]:
import pandas as pd
import numpy as np

print("💼 CASO INTEGRADOR: ANÁLISIS DE COHORTES")
print("="*70)

# Cohortes de usuarios (mes de registro)
np.random.seed(42)
cohortes = pd.DataFrame({
    'Usuario_ID': range(1, 101),
    'Mes_Registro': np.random.choice(['Ene', 'Feb', 'Mar'], 100),
    'Plan': np.random.choice(['Básico', 'Premium'], 100, p=[0.6, 0.4]),
    'Revenue_Mensual': np.random.randint(10, 100, 100),
    'Activo': np.random.choice([True, False], 100, p=[0.92, 0.08])
})

print(f"\n📊 Dataset: {len(cohortes)} usuarios")
print(cohortes.head(10))

print("\n" + "="*70)
print("\n1️⃣  ARPU POR COHORTE")
print("-"*70)

arpu_cohorte = cohortes.groupby('Mes_Registro').agg({
    'Revenue_Mensual': 'mean',
    'Usuario_ID': 'count'
}).rename(columns={'Revenue_Mensual': 'ARPU', 'Usuario_ID': 'Total_Usuarios'}).reset_index()

arpu_cohorte['ARPU'] = arpu_cohorte['ARPU'].round(2)

print(arpu_cohorte)

print("\n" + "-"*70)
print("\n2️⃣  CHURN RATE POR COHORTE")
print("-"*70)

churn_cohorte = cohortes.groupby('Mes_Registro').agg({
    'Usuario_ID': 'count',
    'Activo': lambda x: (~x).sum()  # Contar usuarios inactivos
}).rename(columns={'Usuario_ID': 'Total', 'Activo': 'Churn_Count'}).reset_index()

churn_cohorte['Churn_Rate_%'] = (churn_cohorte['Churn_Count'] / churn_cohorte['Total'] * 100).round(2)

print(churn_cohorte)

print("\n" + "-"*70)
print("\n3️⃣  ARPU POR PLAN")
print("-"*70)

arpu_plan = cohortes.groupby('Plan').agg({
    'Revenue_Mensual': 'mean',
    'Usuario_ID': 'count'
}).rename(columns={'Revenue_Mensual': 'ARPU', 'Usuario_ID': 'Usuarios'}).reset_index()

arpu_plan['ARPU'] = arpu_plan['ARPU'].round(2)

print(arpu_plan)

print("\n" + "-"*70)
print("\n4️⃣  CHURN RATE POR PLAN")
print("-"*70)

churn_plan = cohortes.groupby('Plan').agg({
    'Usuario_ID': 'count',
    'Activo': lambda x: (~x).sum()
}).rename(columns={'Usuario_ID': 'Total', 'Activo': 'Churn_Count'}).reset_index()

churn_plan['Churn_Rate_%'] = (churn_plan['Churn_Count'] / churn_plan['Total'] * 100).round(2)

print(churn_plan)

print("\n" + "-"*70)
print("\n5️⃣  LTV POR PLAN")
print("-"*70)

analisis_plan = arpu_plan.merge(churn_plan[['Plan', 'Churn_Rate_%']], on='Plan')
analisis_plan['LTV'] = (analisis_plan['ARPU'] / (analisis_plan['Churn_Rate_%'] / 100)).round(2)

print(analisis_plan)

print("\n👉 CONCLUSIONES:")
for idx, row in analisis_plan.iterrows():
    plan = row['Plan']
    arpu = row['ARPU']
    churn = row['Churn_Rate_%']
    ltv = row['LTV']
    
    print(f"\n{plan}:")
    print(f"  ARPU: ${arpu:.2f}/mes")
    print(f"  Churn: {churn:.2f}%")
    print(f"  LTV: ${ltv:.2f}")
    
    if ltv > 500:
        print(f"  ✅ Excelente LTV - Cliente muy valioso")
    elif ltv > 300:
        print(f"  🟡 Buen LTV")
    else:
        print(f"  ⚠️ LTV bajo - Mejorar retención")

print("\n" + "="*70)
print("✅ Análisis de cohortes completado")

## 🎓 Conclusiones del Notebook 06_03

### ✅ Lo que aprendiste

1. **ARPU (Average Revenue Per User):**
   ```python
   ARPU = Total_Revenue / Total_Users
   ```

2. **Churn Rate:**
   ```python
   Churn_Rate = (Cancelaciones / Usuarios_Inicio) * 100
   ```

3. **LTV (Lifetime Value):**
   ```python
   LTV = ARPU / Churn_Rate
   ```

4. **Análisis por Cohortes:**
   - Comparar ARPU y Churn por mes de registro
   - Identificar cohortes más valiosas
   - Optimizar estrategias por segmento

---

### 🛠️ Guía de interpretación

**ARPU:**
* 🟢 Alto: Mayor monetización por usuario
* 🟡 Medio: Estándar de la industria
* 🔴 Bajo: Oportunidad de upselling

**Churn Rate:**
* 🟢 < 3%: Excelente retención
* 🟡 3-7%: Aceptable
* 🔴 > 7%: Problema de retención

**LTV:**
* 🟢 > $500: Cliente muy valioso
* 🟡 $200-500: Bueno
* 🔴 < $200: Bajo - Mejorar retención o ARPU

---


<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>📊 ¡ARPU y Churn Rate Dominados!</h3>
  <p><i>"LTV = ARPU / Churn: La fórmula que define el valor de tus clientes."</i></p>
</div>